In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import measure
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [ ]:
ROOT_PATH = "/content/drive/MyDrive/mashob/labaCompetit/recodai-luc-scientific-image-forgery-detection"

try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    print("Google Drive уже смонтирован или монтирование не требуется.")

if not os.path.exists(ROOT_PATH):
    raise FileNotFoundError(f" Директория {ROOT_PATH} не найдена. Убедитесь, что данные загружены в Google Drive.")

Mounted at /content/drive


In [ ]:

IMG_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5

def rle_encode(mask):
    mask = mask.T
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs) if len(runs) > 0 else ""

def is_mask_empty(mask):
    return mask.sum() == 0


# Dataset

class ForgeryDataset(Dataset):
    def __init__(self, image_ids, img_dir, mask_dir=None, transform=None, size=IMG_SIZE):
        self.image_ids = image_ids
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.size = size

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, img_id)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Resize
        h, w = image.shape
        if h != self.size or w != self.size:
            image = cv2.resize(image, (self.size, self.size), interpolation=cv2.INTER_AREA)
        image = image.astype(np.float32) / 255.0
        image = np.expand_dims(image, axis=0)  # (1, H, W)

        if self.mask_dir is not None:
            mask_path = os.path.join(self.mask_dir, img_id.replace(".jpg", ".png").replace(".jpeg", ".png"))
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is None:
                    mask = np.zeros((h, w), dtype=np.uint8)
                else:
                    mask = (mask > 0).astype(np.uint8)
                mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
            else:
                mask = np.zeros((self.size, self.size), dtype=np.uint8)
            mask = np.expand_dims(mask, axis=0).astype(np.float32)
            return torch.from_numpy(image), torch.from_numpy(mask)
        else:
            return torch.from_numpy(image)


# U-Net-like модель

class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 2, stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.enc(x)
        x = self.dec(x)
        return x


# Обучение

def train_model():
    # Сбор ID изображений
    train_ids = [f for f in os.listdir(os.path.join(ROOT_PATH, "train_images")) if f.endswith(('.png', '.jpg', '.jpeg'))]
    supp_ids = [f for f in os.listdir(os.path.join(ROOT_PATH, "supplemental_images")) if f.endswith(('.png', '.jpg', '.jpeg'))]
    all_train_ids = train_ids + supp_ids

    train_img_dir = os.path.join(ROOT_PATH, "train_images")
    supp_img_dir = os.path.join(ROOT_PATH, "supplemental_images")
    train_mask_dir = os.path.join(ROOT_PATH, "train_masks")
    supp_mask_dir = os.path.join(ROOT_PATH, "supplemental_masks")

    # Разделение
    train_ids, val_ids = train_test_split(all_train_ids, test_size=0.1, random_state=42)

    # Датасеты
    train_dataset = ForgeryDataset(
        train_ids,
        img_dir=train_img_dir,
        mask_dir=train_mask_dir,
        size=IMG_SIZE
    )


    class CombinedDataset(Dataset):
        def __init__(self, ids, img_dirs, mask_dirs, size=IMG_SIZE):
            self.ids = ids
            self.img_dirs = img_dirs
            self.mask_dirs = mask_dirs
            self.size = size

        def __len__(self):
            return len(self.ids)

        def __getitem__(self, idx):
            img_id = self.ids[idx]
            # Ищем изображение
            img_path = None
            for d in self.img_dirs:
                p = os.path.join(d, img_id)
                if os.path.exists(p):
                    img_path = p
                    break
            if img_path is None:
                raise FileNotFoundError(f"Image {img_id} not found")

            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                image = cv2.imread(img_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            image = cv2.resize(image, (self.size, self.size), interpolation=cv2.INTER_AREA)
            image = image.astype(np.float32) / 255.0
            image = np.expand_dims(image, 0)

            # Ищем маску
            mask = None
            for d in self.mask_dirs:
                mask_path = os.path.join(d, img_id.replace(".jpg", ".png").replace(".jpeg", ".png"))
                if os.path.exists(mask_path):
                    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                    if m is not None:
                        mask = (m > 0).astype(np.uint8)
                        break
            if mask is None:
                mask = np.zeros((image.shape[1], image.shape[2]), dtype=np.uint8)
            else:
                mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)

            mask = np.expand_dims(mask, 0).astype(np.float32)
            return torch.from_numpy(image), torch.from_numpy(mask)

    train_dataset = CombinedDataset(
        train_ids,
        img_dirs=[train_img_dir, supp_img_dir],
        mask_dirs=[train_mask_dir, supp_mask_dir],
        size=IMG_SIZE
    )
    val_dataset = CombinedDataset(
        val_ids,
        img_dirs=[train_img_dir, supp_img_dir],
        mask_dirs=[train_mask_dir, supp_mask_dir],
        size=IMG_SIZE
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = SimpleUNet().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss = criterion(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Train Loss: {total_loss/len(train_loader):.4f}")

        # Валидация (опционально)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                loss = criterion(preds, masks)
                val_loss += loss.item()
        print(f"Val Loss: {val_loss/len(val_loader):.4f}")

    return model


# Предсказание на test

def predict_test(model):
    test_dir = os.path.join(ROOT_PATH, "test_images")
    test_ids = [f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    test_dataset = ForgeryDataset(test_ids, img_dir=test_dir, mask_dir=None, size=IMG_SIZE)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model.eval()
    results = []
    with torch.no_grad():
        for i, imgs in enumerate(tqdm(test_loader, desc="Predicting")):
            imgs = imgs.to(DEVICE)
            preds = model(imgs)
            preds = (preds > THRESHOLD).cpu().numpy()

            for j in range(preds.shape[0]):
                mask = preds[j, 0]  # (H, W)
                if is_mask_empty(mask):
                    annotation = "authentic"
                else:
                    annotation = rle_encode(mask.astype(np.uint8))
                case_id = test_ids[i * BATCH_SIZE + j]
                results.append({"case_id": case_id, "annotation": annotation})

    df = pd.DataFrame(results)
    df.to_csv(os.path.join(ROOT_PATH, "submission.csv"), index=False)
    print("Submission saved to submission.csv")


if __name__ == "__main__":
    model = train_model()
    predict_test(model)

Epoch 1/5:   0%|          | 0/6 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 1/5: 100%|██████████| 6/6 [00:53<00:00,  8.96s/it]

Train Loss: 0.5870


Val Loss: 0.4798


Epoch 2/5: 100%|██████████| 6/6 [00:38<00:00,  6.34s/it]

Train Loss: 0.3324


...

Epoch 5/5: 100%|██████████| 6/6 [00:37<00:00,  6.18s/it]

Train Loss: 0.0002


Val Loss: 0.0001


Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Submission saved to submission.csv


In [ ]:
import matplotlib.pyplot as plt

IMG_SIZE = 512
BATCH_SIZE = 50
EPOCHS = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5


def rle_encode(mask):
    mask = mask.T
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs) if len(runs) > 0 else ""

def is_mask_empty(mask):
    return mask.sum() == 0


class ForgeryDataset(Dataset):
    def __init__(self, image_ids, img_dir, mask_dir=None, transform=None, size=IMG_SIZE):
        self.image_ids = image_ids
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.size = size

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, img_id)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            image = cv2.imread(img_path)  # fallback to color
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Resize
        h, w = image.shape
        if h != self.size or w != self.size:
            image = cv2.resize(image, (self.size, self.size), interpolation=cv2.INTER_AREA)
        image = image.astype(np.float32) / 255.0
        image = np.expand_dims(image, axis=0)  # (1, H, W)

        if self.mask_dir is not None:
            mask_path = os.path.join(self.mask_dir, img_id.replace(".jpg", ".png").replace(".jpeg", ".png"))
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if mask is None:
                    mask = np.zeros((h, w), dtype=np.uint8)
                else:
                    mask = (mask > 0).astype(np.uint8)
                mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
            else:
                mask = np.zeros((self.size, self.size), dtype=np.uint8)
            mask = np.expand_dims(mask, axis=0).astype(np.float32)
            return torch.from_numpy(image), torch.from_numpy(mask)
        else:
            return torch.from_numpy(image)

# U-Net-like модель
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 2, stride=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.enc(x)
        x = self.dec(x)
        return x


# Обучение

def train_model():

    train_ids = [f for f in os.listdir(os.path.join(ROOT_PATH, "train_images")) if f.endswith(('.png', '.jpg', '.jpeg'))]
    supp_ids = [f for f in os.listdir(os.path.join(ROOT_PATH, "supplemental_images")) if f.endswith(('.png', '.jpg', '.jpeg'))]
    all_train_ids = train_ids + supp_ids

    train_img_dir = os.path.join(ROOT_PATH, "train_images")
    supp_img_dir = os.path.join(ROOT_PATH, "supplemental_images")
    train_mask_dir = os.path.join(ROOT_PATH, "train_masks")
    supp_mask_dir = os.path.join(ROOT_PATH, "supplemental_masks")

    # Разделение
    train_ids, val_ids = train_test_split(all_train_ids, test_size=0.1, random_state=42)

    # Датасеты
    train_dataset = ForgeryDataset(
        train_ids,
        img_dir=train_img_dir,
        mask_dir=train_mask_dir,
        size=IMG_SIZE
    )


    class CombinedDataset(Dataset):
        def __init__(self, ids, img_dirs, mask_dirs, size=IMG_SIZE):
            self.ids = ids
            self.img_dirs = img_dirs
            self.mask_dirs = mask_dirs
            self.size = size

        def __len__(self):
            return len(self.ids)

        def __getitem__(self, idx):
            img_id = self.ids[idx]
            # Ищем изображение
            img_path = None
            for d in self.img_dirs:
                p = os.path.join(d, img_id)
                if os.path.exists(p):
                    img_path = p
                    break
            if img_path is None:
                raise FileNotFoundError(f"Image {img_id} not found")

            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                image = cv2.imread(img_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            image = cv2.resize(image, (self.size, self.size), interpolation=cv2.INTER_AREA)
            image = image.astype(np.float32) / 255.0
            image = np.expand_dims(image, 0)

            # Ищем маску
            mask = None
            for d in self.mask_dirs:
                mask_path = os.path.join(d, img_id.replace(".jpg", ".png").replace(".jpeg", ".png"))
                if os.path.exists(mask_path):
                    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                    if m is not None:
                        mask = (m > 0).astype(np.uint8)
                        break
            if mask is None:
                mask = np.zeros((image.shape[1], image.shape[2]), dtype=np.uint8)
            else:
                mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)

            mask = np.expand_dims(mask, 0).astype(np.float32)
            return torch.from_numpy(image), torch.from_numpy(mask)

    train_dataset = CombinedDataset(
        train_ids,
        img_dirs=[train_img_dir, supp_img_dir],
        mask_dirs=[train_mask_dir, supp_mask_dir],
        size=IMG_SIZE
    )
    val_dataset = CombinedDataset(
        val_ids,
        img_dirs=[train_img_dir, supp_img_dir],
        mask_dirs=[train_mask_dir, supp_mask_dir],
        size=IMG_SIZE
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = SimpleUNet().to(DEVICE)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss = criterion(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Train Loss: {total_loss/len(train_loader):.4f}")

        # Валидация (опционально)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                loss = criterion(preds, masks)
                val_loss += loss.item()
        print(f"Val Loss: {val_loss/len(val_loader):.4f}")

    return model


def predict_test(model):
    test_dir = os.path.join(ROOT_PATH, "test_images")
    test_ids = [f for f in os.listdir(test_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
    test_dataset = ForgeryDataset(test_ids, img_dir=test_dir, mask_dir=None, size=IMG_SIZE)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model.eval()
    results = []
    with torch.no_grad():
        for i, imgs in enumerate(tqdm(test_loader, desc="Predicting")):
            imgs = imgs.to(DEVICE)
            preds = model(imgs)
            preds = (preds > THRESHOLD).cpu().numpy()

            for j in range(preds.shape[0]):
                mask = preds[j, 0]  # (H, W)
                if is_mask_empty(mask):
                    annotation = "authentic"
                else:
                    annotation = "false"
                case_id = test_ids[i * BATCH_SIZE + j]
                results.append({"case_id": case_id, "annotation": annotation})

    df = pd.DataFrame(results)
    df.to_csv(os.path.join(ROOT_PATH, "submission3.csv"), index=False)
    print("Submission saved to submission3.csv")


def visualize_predictions(model, img_dir, mask_dir, image_ids, num_samples=5, size=IMG_SIZE):
    """
    Визуализирует предсказания модели на нескольких изображениях из тренировочного набора.
    """
    model.eval()
    shown = 0
    with torch.no_grad():
        for img_id in image_ids:
            if shown >= num_samples:
                break
            # Загрузка изображения
            img_path = os.path.join(img_dir, img_id)
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                image = cv2.imread(img_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            orig_h, orig_w = image.shape
            image_resized = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
            image_tensor = torch.from_numpy(image_resized.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)

            # Предсказание
            pred_mask = model(image_tensor).cpu().numpy()[0, 0]
            pred_mask = (pred_mask > THRESHOLD).astype(np.uint8)

            # Загрузка истинной маски
            mask_path = os.path.join(mask_dir, img_id.replace(".jpg", ".png").replace(".jpeg", ".png"))
            if os.path.exists(mask_path):
                true_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if true_mask is not None:
                    true_mask = (true_mask > 0).astype(np.uint8)
                    true_mask = cv2.resize(true_mask, (size, size), interpolation=cv2.INTER_NEAREST)
                else:
                    true_mask = np.zeros((size, size), dtype=np.uint8)
            else:
                true_mask = np.zeros((size, size), dtype=np.uint8)

            # Визуализация
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(image_resized, cmap='gray')
            axes[0].set_title("Input Image")
            axes[0].axis('off')

            axes[1].imshow(true_mask, cmap='gray')
            axes[1].set_title("Ground Truth Mask")
            axes[1].axis('off')

            axes[2].imshow(image_resized, cmap='gray')
            axes[2].imshow(pred_mask, alpha=0.5, cmap='jet')
            axes[2].set_title("Predicted Mask")
            axes[2].axis('off')

            plt.tight_layout()
            plt.show()

            shown += 1

if __name__ == "__main__":
    model = train_model()


    print("\n🔍 Визуализация предсказаний на тренировочной выборке...")
    train_img_dir = os.path.join(ROOT_PATH, "train_images")
    train_mask_dir = os.path.join(ROOT_PATH, "train_masks")
    train_ids = [f for f in os.listdir(train_img_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    sample_ids = train_ids[:5]
    visualize_predictions(model, train_img_dir, train_mask_dir, sample_ids, num_samples=5)


    predict_test(model)

Epoch 1/8: 100%|██████████| 1/1 [00:37<00:00, 37.80s/it]

Train Loss: 0.4907


Val Loss: 0.4819


Epoch 2/8: 100%|██████████| 1/1 [00:37<00:00, 37.19s/it]

Train Loss: 0.4815


...

Epoch 8/8: 100%|██████████| 1/1 [00:36<00:00, 36.39s/it]

Train Loss: 0.3160


Val Loss: 0.2817

🔍 Визуализация предсказаний на тренировочной выборке...


Predicting: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]

Submission saved to submission3.csv
